In [1]:
import sys
import os
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm

# Ensure the notebook can find your src folder modules
sys.path.append(os.path.abspath("."))

from src.ttppr.stage1_dl.dataset import ClinicalICDDataset
from src.ttppr.stage1_dl.model import LAAT

# Hardware check
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

PyTorch Version: 2.13.0+cpu
CUDA Available: False


In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for Multi-Label Classification to address severe class imbalance."""
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce_with_logits = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, inputs, targets):
        bce_loss = self.bce_with_logits(inputs, targets)
        pt = torch.exp(-bce_loss) 
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        
        if self.reduction == "mean":
            return focal_loss.mean()
        return focal_loss.sum()

In [ ]:
# --- HYPERPARAMETERS ---
EPOCHS = 100
BATCH_SIZE = 32  # You can increase this to 64 or 128 depending on Supercomputer VRAM
LEARNING_RATE = 3e-5
SAVE_PATH = "data/processed/stage1_laat_supercomputer_100ep.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load and Split Dataset (80% Train, 20% Validation)
print("Loading dataset...")
full_dataset = ClinicalICDDataset(
    notes_csv="data/raw/mimic/discharge.csv",
    labels_csv="data/raw/mimic/diagnosis.csv",
    code_map_path="data/processed/code_vocab.json",
)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# pin_memory=True speeds up CPU-to-GPU data transfer
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"Training samples: {train_size} | Validation samples: {val_size}")

# 2. Initialize Model
print("Initializing BioClinical-BERT + LAAT...")
model = LAAT(
    model_name="emilyalsentzer/Bio_ClinicalBERT",
    num_classes=full_dataset.num_classes, 
)

# 3. MULTI-GPU ACCELERATION
if torch.cuda.device_count() > 1:
    print(f"Activating DataParallel across {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model = model.to(device)

# Unfreeze all layers for a deep 100-epoch run
for param in model.parameters():
    param.requires_grad = True

criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# 4. Training Loop with Best-Model Checkpointing based on VALIDATION Loss
print(f"\nStarting {EPOCHS}-Epoch Supercomputer Training...")
best_val_loss = float('inf')

Path(SAVE_PATH).parent.mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):
    
    # ==========================
    #      TRAIN PHASE
    # ==========================
    model.train()
    running_train_loss = 0.0
    
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    
    for batch in train_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)

        optimizer.zero_grad()
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_train_loss += loss.item()
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = running_train_loss / len(train_loader)
    
    # ==========================
    #    VALIDATION PHASE
    # ==========================
    model.eval()
    running_val_loss = 0.0
    
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)
        for batch in val_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, targets)
            running_val_loss += loss.item()
            
    avg_val_loss = running_val_loss / len(val_loader)
    
    # ==========================
    #     CHECKPOINTING
    # ==========================
    print(f"Epoch {epoch+1}/{EPOCHS} Complete | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Save the model ONLY if Validation Loss improves
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        
        # If using DataParallel, extract the base module to save correctly
        state_dict_to_save = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        
        torch.save(state_dict_to_save, SAVE_PATH)
        print(f"   -> Validation improved! Saved best checkpoint to '{SAVE_PATH}'")

print("\n[✔] 100-Epoch Supercomputer Training Complete!")

Loading dataset...


Initializing BioClinical-BERT + LAAT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Starting 100-Epoch Training...


Epoch 1/100:   0%|          | 0/320 [00:00<?, ?it/s]

c:\courses\TTPPR\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
